In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots as sp
plt.style.use('science')
plt.rcParams['font.size'] = 18
import sys

In [ ]:
sys.path.append('../')
from preprocessing.ng_loader import df_all_cycles, generate_row_script
from preprocessing.ng_cycles_stats import calculate_cycles
from data_generation.synthetic_script import generate_full_script
from scipy.stats import binomtest, kstest

In [3]:
df_synthetic = generate_full_script(n_samples=1000, time_unit='months', percent_idle=0.13)

df_all = df_all_cycles('../data/agsi_depleted/', MAXrate=6 *(10**5), time_unit='months', synthetic=False)

Inj_samples mean: 5.079416666666667, std: 2.44041178077025, median: 6.0
With_samples mean: 4.777694444444444, std: 2.0032049919988557, median: 5.0
Idle_samples mean: 2.4137386131251164, std: 1.210870138066204, median: 2.0
Normalized ___________________________
Inj_samples mean: 0.40753514893267573, std: 0.2412656925988787, median: 0.47549397722355435
With_samples mean: 0.41973916962879143, std: 0.2179017466276247, median: 0.46533839626617
Idle_samples mean: 0.2827507189924687, std: 0.2280240526257908, median: 0.20002871314485818


/home/tlp25/irp-tlp25/Report_plots/../Generation_data_scripts/synthetic_script.py:259: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.80776642 0.63557437 0.97311179 ... 0.47189557 0.79353169 0.77652248]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[(df['Phase'] == 'injection') &


In [4]:
df_all

,Country,Site,Month,Injection_GWh_d,Withdrawal_GWh_d,Gas_in_storage_TWh,Full_percent,Technical_Capacity_TWh,Injection_capacity_GWh/d,Withdrawal_capacity_GWh_d,...,Available_Capacity_TWh,Phase,Net_Injection_Withdrawal_GWh_d,Normalised_Net_Injection_Withdrawal,Phase_Change,Cycle_ID,Row_Script,Cycle_position,Normalised_cycle_pos,Normalised_cycle_pos_cut
0,Austria,UGS Haidach,2015-05,44.879545,0.000000,13.6149,69.69,19.536,177.6,195.36,...,0.0,injection,44.879545,0.662814,1,1,WCONINJE\n WELL3 GAS 1* RATE 397688.2873372228...,1,0.2,0.33
1,Austria,UGS Haidach,2015-06,46.427000,0.000000,15.0077,76.82,19.536,177.6,195.36,...,0.0,injection,46.427000,0.667293,0,1,WCONINJE\n WELL3 GAS 1* RATE 400375.9592273516...,2,0.4,0.66
2,Austria,UGS Haidach,2015-07,32.578710,0.000000,16.0176,81.99,19.536,177.6,195.36,...,0.0,injection,32.578710,0.627206,0,1,WCONINJE\n WELL3 GAS 1* RATE 376323.7757653743...,3,0.6,0.66
3,Austria,UGS Haidach,2015-08,39.551613,31.387097,16.2708,83.29,19.536,177.6,195.36,...,0.0,injection,8.164516,0.556534,0,1,WCONINJE\n WELL3 GAS 1* RATE 333920.3708234275...,4,0.8,1.00
4,Austria,UGS Haidach,2015-09,61.406333,11.396667,17.7712,90.97,19.536,177.6,195.36,...,0.0,injection,50.009667,0.677664,0,1,WCONINJE\n WELL3 GAS 1* RATE 406598.4570189576...,5,1.0,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4324,United Kingdom,UGS Humbly Grove,2026-02,0.000000,13.646429,0.2714,9.64,2.815,89.0,20.00,...,0.0,withdrawal,-13.646429,0.380769,0,992,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 228461.4...,4,1.0,1.00
4325,United Kingdom,UGS Humbly Grove,2026-03,0.000000,0.000000,0.2714,9.64,2.815,89.0,20.00,...,0.0,idle,0.000000,0.504643,1,993,WELLSHUT\n'WELL3' /\n /\nTSTEP\n30 /\n,1,1.0,1.00
4326,United Kingdom,UGS Humbly Grove,2026-04,0.000000,3.826667,0.1840,6.54,2.815,89.0,20.00,...,0.0,withdrawal,-3.826667,0.469907,1,994,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 281943.9...,1,0.5,0.66
4327,United Kingdom,UGS Humbly Grove,2026-05,0.000000,5.245161,0.0211,0.75,2.815,89.0,9.00,...,0.0,withdrawal,-5.245161,0.457030,0,994,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 274218.2...,2,1.0,1.00


In [5]:
inj_final_cycle_idxs = df_all[df_all['Phase'] == 'injection'].groupby(['Site', 'Cycle_ID'])['Cycle_position'].idxmax()
with_final_cycle_idxs = df_all[df_all['Phase'] == 'withdrawal'].groupby(['Site', 'Cycle_ID'])['Cycle_position'].idxmax()

In [6]:
idle = 0
withdrawal = 0


for idx in inj_final_cycle_idxs:
    cycle = df_all.loc[idx]
    next_idx = df_all.loc[idx+1]
    if next_idx['Phase'] == 'idle':
        idle += 1
    elif next_idx['Phase'] == 'withdrawal':
        withdrawal += 1

idle_with = 0
inj_with = 0

for idx in with_final_cycle_idxs:
    cycle = df_all.loc[idx]
    next_idx = df_all.loc[idx+1]
    if next_idx['Phase'] == 'idle':
        idle_with += 1
    elif next_idx['Phase'] == 'injection':
        inj_with += 1


In [7]:
idle, withdrawal, len(inj_final_cycle_idxs)

(60, 371, 435)

In [8]:
idle_with, inj_with, len(with_final_cycle_idxs)

(56, 388, 444)

In [9]:
num_idle = df_all[df_all['Phase'] == 'idle'].shape[0]

num_idle, len(df_all)

(230, 4329)

In [10]:
56/444

0.12612612612612611

In [11]:
60/435

0.13793103448275862

In [12]:
k = 60
n = 435
p= num_idle / len(df_all)

binomtest(k, n, p, alternative='greater')

BinomTestResult(k=60, n=435, alternative='greater', statistic=0.13793103448275862, pvalue=2.2198812609890067e-11)

In [13]:
df_all

,Country,Site,Month,Injection_GWh_d,Withdrawal_GWh_d,Gas_in_storage_TWh,Full_percent,Technical_Capacity_TWh,Injection_capacity_GWh/d,Withdrawal_capacity_GWh_d,...,Available_Capacity_TWh,Phase,Net_Injection_Withdrawal_GWh_d,Normalised_Net_Injection_Withdrawal,Phase_Change,Cycle_ID,Row_Script,Cycle_position,Normalised_cycle_pos,Normalised_cycle_pos_cut
0,Austria,UGS Haidach,2015-05,44.879545,0.000000,13.6149,69.69,19.536,177.6,195.36,...,0.0,injection,44.879545,0.662814,1,1,WCONINJE\n WELL3 GAS 1* RATE 397688.2873372228...,1,0.2,0.33
1,Austria,UGS Haidach,2015-06,46.427000,0.000000,15.0077,76.82,19.536,177.6,195.36,...,0.0,injection,46.427000,0.667293,0,1,WCONINJE\n WELL3 GAS 1* RATE 400375.9592273516...,2,0.4,0.66
2,Austria,UGS Haidach,2015-07,32.578710,0.000000,16.0176,81.99,19.536,177.6,195.36,...,0.0,injection,32.578710,0.627206,0,1,WCONINJE\n WELL3 GAS 1* RATE 376323.7757653743...,3,0.6,0.66
3,Austria,UGS Haidach,2015-08,39.551613,31.387097,16.2708,83.29,19.536,177.6,195.36,...,0.0,injection,8.164516,0.556534,0,1,WCONINJE\n WELL3 GAS 1* RATE 333920.3708234275...,4,0.8,1.00
4,Austria,UGS Haidach,2015-09,61.406333,11.396667,17.7712,90.97,19.536,177.6,195.36,...,0.0,injection,50.009667,0.677664,0,1,WCONINJE\n WELL3 GAS 1* RATE 406598.4570189576...,5,1.0,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4324,United Kingdom,UGS Humbly Grove,2026-02,0.000000,13.646429,0.2714,9.64,2.815,89.0,20.00,...,0.0,withdrawal,-13.646429,0.380769,0,992,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 228461.4...,4,1.0,1.00
4325,United Kingdom,UGS Humbly Grove,2026-03,0.000000,0.000000,0.2714,9.64,2.815,89.0,20.00,...,0.0,idle,0.000000,0.504643,1,993,WELLSHUT\n'WELL3' /\n /\nTSTEP\n30 /\n,1,1.0,1.00
4326,United Kingdom,UGS Humbly Grove,2026-04,0.000000,3.826667,0.1840,6.54,2.815,89.0,20.00,...,0.0,withdrawal,-3.826667,0.469907,1,994,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 281943.9...,1,0.5,0.66
4327,United Kingdom,UGS Humbly Grove,2026-05,0.000000,5.245161,0.0211,0.75,2.815,89.0,9.00,...,0.0,withdrawal,-5.245161,0.457030,0,994,WCONPROD\n WELL3 OPEN WGRA 5* 96.5 3* 274218.2...,2,1.0,1.00


In [21]:
def summary_stats(df, cycle_position):
    df_pos_rates = df[df['Normalised_cycle_pos_cut'] == cycle_position]
    print(f"Summary statistics for cycle position Injection Rate Position {cycle_position}:")
    df_pos_rates_inj = df_pos_rates[df_pos_rates['Phase'] == 'injection']['Normalised_Net_Injection_Withdrawal']
    print(f"Mean: {df_pos_rates_inj.mean()}")
    print(f"Median: {df_pos_rates_inj.median()}")
    print(f"Standard Deviation: {df_pos_rates_inj.std()}")
    df_pos_rates_with = df_pos_rates[df_pos_rates['Phase'] == 'withdrawal']['Normalised_Net_Injection_Withdrawal']
    print(f"Summary statistics for cycle position Withdrawal Rate Position {cycle_position}:")
    print(f"Mean: {df_pos_rates_with.mean()}")
    print(f"Median: {df_pos_rates_with.median()}")
    print(f"Standard Deviation: {df_pos_rates_with.std()}")
    return None

In [22]:
summary_stats(df_all, 1)

Summary statistics for cycle position Injection Rate Position 1:
Mean: 0.7094702141373801
Median: 0.6973876040048648
Standard Deviation: 0.1375062768865727
Summary statistics for cycle position Withdrawal Rate Position 1:
Mean: 0.3821167120078096
Median: 0.38867045547411333
Standard Deviation: 0.17557187629266197


In [23]:
summary_stats(df_all, 0.66)

Summary statistics for cycle position Injection Rate Position 0.66:
Mean: 0.7666141843965376
Median: 0.7696102846774571
Standard Deviation: 0.1375309004028339
Summary statistics for cycle position Withdrawal Rate Position 0.66:
Mean: 0.30819308444108395
Median: 0.2970029637333605
Standard Deviation: 0.18364278787743094


In [24]:
summary_stats(df_all, 0.33)

Summary statistics for cycle position Injection Rate Position 0.33:
Mean: 0.6957535496519517
Median: 0.6800308907297301
Standard Deviation: 0.12949399641330456
Summary statistics for cycle position Withdrawal Rate Position 0.33:
Mean: 0.42507454466296063
Median: 0.43404195301831966
Standard Deviation: 0.15656059156348712


In [ ]:
def ks_test(df, cycle_pos1, cycle_pos2):
    df_pos1 = df[df['Normalised_cycle_pos_cut'] == cycle_pos1]
    df_pos2 = df[df['Normalised_cycle_pos_cut'] == cycle_pos2]
    df_pos1_inj = df_pos1[df_pos1['Phase'] == 'injection']['Normalised_Net_Injection_Withdrawal']
    df_pos2_inj = df_pos2[df_pos2['Phase'] == 'injection']['Normalised_Net_Injection_Withdrawal']
    df_pos1_with = df_pos1[df_pos1['Phase'] == 'withdrawal']['Normalised_Net_Injection_Withdrawal']
    df_pos2_with = df_pos2[df_pos2['Phase'] == 'withdrawal']['Normalised_Net_Injection_Withdrawal']

    ks_stat_inj, p_value_inj = kstest(df_pos1_inj, df_pos2_inj, alternative='two-sided')
    ks_stat_with, p_value_with = kstest(df_pos1_with, df_pos2_with, alternative='two-sided')

    print(f"KS test for Injection Rate Position {cycle_pos1} vs {cycle_pos2}:")
    print(f"KS Statistic: {ks_stat_inj}, P-value: {p_value_inj}")

    print(f"KS test for Withdrawal Rate Position {cycle_pos1} vs {cycle_pos2}:")
    print(f"KS Statistic: {ks_stat_with}, P-value: {p_value_with}")

    return None

In [34]:
ks_test(df_all, 1, 0.33)

KS test for Injection Rate Position 1 vs 0.33:
KS Statistic: 0.0699317738791423, P-value: 0.08693631654493322
KS test for Withdrawal Rate Position 1 vs 0.33:
KS Statistic: 0.14116580698213352, P-value: 7.843049156014188e-05


In [35]:
ks_test(df_all, 1, 0.66)

KS test for Injection Rate Position 1 vs 0.66:
KS Statistic: 0.20520317888738943, P-value: 7.648635063909791e-16
KS test for Withdrawal Rate Position 1 vs 0.66:
KS Statistic: 0.18355910337042414, P-value: 1.2414716533329051e-11


In [36]:
ks_test(df_all, 0.66, 0.33)

KS test for Injection Rate Position 0.66 vs 0.33:
KS Statistic: 0.23384315489578647, P-value: 9.371854462029126e-14
KS test for Withdrawal Rate Position 0.66 vs 0.33:
KS Statistic: 0.30813484423418963, P-value: 3.2925246307887587e-19
